# Train the model

Install all dependencies

In [ ]:
!pip install torch transformers

Download train dataset from GitHub Repository

In [ ]:
!wget "https://github.com/joaompfonseca/ri-neural-reranker/raw/main/data/train_dataset.jsonl" -O train_dataset.jsonl

Import all modules

In [ ]:
import gzip
import json
import math
import torch

from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, BertConfig, BertForSequenceClassification
from sklearn.metrics import precision_score, recall_score

Check that the dataset is correctly downloaded

In [ ]:
import json

with open("train_dataset.jsonl") as train_dataset:
  for line in train_dataset:
    data = json.loads(line)
    print(data["query"])
    print(data["pos_docs"])
    print(data["neg_docs"])
    break

Define the model checkpoint and device

In [ ]:
model_checkpoint = "bert-base-uncased"

Choose the device

In [ ]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

Define the model

In [ ]:
class NeuralReranker(BertForSequenceClassification):

  def __init__(self, checkpoint):
    config = BertConfig.from_pretrained(checkpoint)
    super().__init__(config)

    # Initialize the BERT model from checkpoint
    self.bert = self.bert.from_pretrained(checkpoint)

    # Additional layers for reranking
    self.dropout = torch.nn.Dropout(p=0.1)
    self.linear1 = torch.nn.Linear(config.hidden_size, 1024)
    self.act = torch.nn.GELU()
    self.linear2 = torch.nn.Linear(1024, 2)

  def forward(self,
            input_ids=None,
            attention_mask=None,
            token_type_ids=None,
            position_ids=None,
            indexes=None,
            novel=None,
            head_mask=None,
            inputs_embeds=None,
            labels=None,
            output_attentions=None,
            output_hidden_states=None,
            return_dict=None,
            mask=None,
           ):

    x = self.bert(input_ids,
                  attention_mask=attention_mask,
                  token_type_ids=token_type_ids,
                  position_ids=position_ids,
                  head_mask=head_mask,
                  inputs_embeds=inputs_embeds,
                  output_attentions=output_attentions,
                  output_hidden_states=output_hidden_states,
                  return_dict=return_dict)

    x = self.dropout(x["pooler_output"])
    x = self.linear1(x)
    x = self.act(x)
    x = self.linear2(x)
    return x

Create DataLoaders for the train and evaluation datasets

In [ ]:
TRAIN_PERCENT = 0.05
EVAL_PERCENT = 0.01
BATCH_SIZE = 16

In [ ]:
class CustomDataset(Dataset):
  def __init__(self, queries, pos_docs, neg_docs):
    self.queries = queries
    self.pos_docs = pos_docs
    self.neg_docs = neg_docs

  def __len__(self):
    return len(self.queries)

  def __getitem__(self, idx):
    return self.queries[idx], self.pos_docs[idx], self.neg_docs[idx]

# Transform the dataset into single entries
queries = []; pos_docs = []; neg_docs = []; n_entries = 0
with open("train_dataset.jsonl") as train_dataset:
  for line in train_dataset:
    data = json.loads(line)
    N = len(data["pos_docs"])
    for i in range(N):
      query = data["query"]
      pos_doc = data["pos_docs"][i]["text"]
      neg_doc = data["neg_docs"][i]["text"]

      queries.append(query)
      pos_docs.append(pos_doc)
      neg_docs.append(neg_doc)
      n_entries += 1

print(f"Created {n_entries} entries")

# Initialize the Dataset and DataLoader

TRAIN_SIZE = math.floor(n_entries * TRAIN_PERCENT)
print(f"Choosing {TRAIN_SIZE} entries for training dataset ({TRAIN_PERCENT * 100:.2f}% of entries)")

train_dataset = CustomDataset(
    queries[:TRAIN_SIZE],
    pos_docs[:TRAIN_SIZE],
    neg_docs[:TRAIN_SIZE]
)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE)

EVAL_SIZE = math.floor(n_entries * EVAL_PERCENT)
print(f"Choosing {EVAL_SIZE} entries for training dataset ({EVAL_PERCENT * 100:.2f}% of entries)")

eval_dataset = CustomDataset(
    queries[TRAIN_SIZE:EVAL_SIZE],
    pos_docs[TRAIN_SIZE:EVAL_SIZE],
    neg_docs[TRAIN_SIZE:EVAL_SIZE]
)
eval_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE)

Configure tokenizer

In [ ]:
TOKENIZER_LENGTH = 128
LEARNING_RATE    = 1e-4
NUMBER_OF_EPOCHS = 5

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.model_max_length = TOKENIZER_LENGTH

def tokenize(data):
  return tokenizer.batch_encode_plus(
      data,
      padding="max_length",
      truncation=True,
      max_length=TOKENIZER_LENGTH,
      return_tensors="pt"
  )

In [ ]:
model = NeuralReranker(model_checkpoint).to(DEVICE)     # Instanciate the model
loss_func = torch.nn.MarginRankingLoss()                # Loss function
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE) # Adam optimizer

Train the model!

In [ ]:
for epoch in range(NUMBER_OF_EPOCHS):

  print(f"Epoch [{epoch + 1}/{NUMBER_OF_EPOCHS}]")

  """ Train """
  model.train()

  train_loss = 0.0

  for queries, pos_docs, neg_docs in train_dataloader:

    optimizer.zero_grad()

    # Encode query with positive and negative documents
    pos_encoded = tokenize(list(zip(queries, pos_docs))).to(DEVICE)
    neg_encoded = tokenize(list(zip(queries, neg_docs))).to(DEVICE)

    # Forward pass
    pos_logits = model(**pos_encoded)
    neg_logits = model(**neg_encoded)

    # Calculate pairwise ranking loss
    target = torch.ones_like(pos_logits)
    loss = loss_func(pos_logits, neg_logits, target)

    # Backward pass and optimization step
    loss.backward()
    optimizer.step()
    train_loss += loss.item()

  print(f"Train - Avg. Loss: {train_loss / len(train_dataloader)}")

  """ Evaluate """
  model.eval()

  eval_loss = 0.0
  all_true_labels = []
  all_pred_labels = []

  with torch.no_grad():
    for queries, pos_docs, neg_docs in eval_dataloader:

      # Encode query with positive and negative documents
      pos_encoded = tokenize(list(zip(queries, pos_docs))).to(DEVICE)
      neg_encoded = tokenize(list(zip(queries, neg_docs))).to(DEVICE)

      # Forward pass
      pos_logits = model(**pos_encoded)
      neg_logits = model(**neg_encoded)

      # Calculate pairwise ranking loss
      target = torch.ones_like(pos_logits)
      loss = loss_func(pos_logits, neg_logits, target)
      eval_loss += loss.item()

      # Binary prediction
      batch_pred = (pos_logits > neg_logits).cpu().numpy()
      all_pred_labels.extend([1 if all(pred) else 0 for pred in batch_pred])
      all_true_labels.extend([1] * len(queries))

  print(f"Eval - Avg. Loss: {eval_loss / len(eval_dataloader)}")

  # Calculate precision and recall
  precision = precision_score(all_true_labels, all_pred_labels)
  recall = recall_score(all_true_labels, all_pred_labels)

  print(f"Precision: {precision}, Recall: {recall}")

torch.save(model.state_dict(), "neural_reranker.pth")

# Rerank BM25 using the model

Download the model, collections and BM25 runs

In [ ]:
!wget "https://uapt33090-my.sharepoint.com/:u:/g/personal/joao_fonseca_ua_pt/EQ9m_soJsYxMgOCQdaQdb6sBCZyfK6QFjtxQeMiyWEK9aA?e=XWChri&download=1" -O neural_reranker.pth

!wget "https://uapt33090-my.sharepoint.com/:u:/g/personal/joao_fonseca_ua_pt/EaBHP-PszZhBr3f4dfAFA3MBAE_XTB6k-iW4mgUf5dYjbg?e=GFuQLT&download=1" -O pubmed_2022_tiny.jsonl.gz
!wget "https://uapt33090-my.sharepoint.com/:u:/g/personal/joao_fonseca_ua_pt/EXZIa3anvQ5DmTZ9MspBspMB6IIRORb1Wrb_a9lTb3GbIA?e=Ngdgai&download=1" -O pubmed_2022_small.jsonl.gz

!wget "https://github.com/joaompfonseca/ri-neural-reranker/raw/main/data/BM25_E9B1.jsonl" -O BM25_E9B1.jsonl
!wget "https://github.com/joaompfonseca/ri-neural-reranker/raw/main/data/BM25_E9B2.jsonl" -O BM25_E9B2.jsonl

Load the model

In [ ]:
model = NeuralReranker(model_checkpoint)
model.load_state_dict(torch.load("neural_reranker.pth", map_location=torch.device(DEVICE)))

Load BM25_E9B1 documents from the tiny dataset

In [ ]:
all_docs = {}
with gzip.open("pubmed_2022_tiny.jsonl.gz", "r") as all_file:
    for i, line in enumerate(all_file):
        print(f"Loading {i} documents from tiny dataset...", end="\r")
        doc = json.loads(line)
        all_docs[doc["pmid"]] = doc["title"] + " " + doc["abstract"]

print()

bm25_runs = []
bm25_documents = dict() # Documents from all_docs that appear in this BM25 run

with open("BM25_E9B1.jsonl") as bm25_tiny:
  for line in bm25_tiny:
    data = json.loads(line)
    bm25_runs += [data]
    for doc in data["documents"]:
      bm25_documents[doc["id"]] = all_docs[doc["id"]]

Rank BM_25 results using the model!

In [ ]:
all_bm25_reranks = []
with torch.no_grad():
  for run in bm25_runs:

    query = run["question"]
    bm25_rerank = []

    for doc in run["documents"]:
      document = bm25_documents[doc["id"]]

      # Encode query with document
      encode = tokenize([f"{query} [SEP] {document}"]).to(DEVICE)
      # Forward pass
      logits = model(**encode)
      # Relevancy score [0-1]
      score = torch.nn.functional.softmax(logits, dim=-1)[:,1]

      bm25_rerank += [{"id": doc["id"], "score": score.item()}]

    bm25_rerank
    break # remove

In [ ]:
bm25_rerank.sort(key=lambda x:-x["score"])
bm25_rerank